In [1]:
from ismn.interface import ISMN_Interface
import geopandas as gpd
from getpass import getuser
import pandas as pd
from pathlib import Path
import numpy as np

## Selected stations

In [2]:
selection_pairs = {
    "ROMPS": ["AbramovGlacier", "Asai", "Baytik", "GolubinGlacier", "Kokomeren", "Maidanak", "Maydantal", "Taragay"],
    "SCAN": ["Kainaliu", "Kukuihaele", "ManaHouse", "PuuwaiRanch", "SilverSword", "WaimeaPlain"],
    "SOILSCAPE": ["Ararimu-2602", "Ararimu-2603", "Wharekapua-2801", "Wharekapua-2805"],
    "YJG-SM": ["P1", "P2", "P3", "P4", "P6", "P7", "P8", "P9", "P10", "P11"]
}

station_list = [item for sublist in selection_pairs.values() for item in sublist]

## Read the ISMN text data

In [3]:
HOME_DIR = Path(f"/home/{getuser()}")
ISMN_DIR = HOME_DIR / "Downloads" / "ISMN_archive_20260922"

WRITE_DIR = Path("./").resolve().parent / "ISMN_data"

TEMPORAL_DIR = WRITE_DIR / "temporal"
TEMPORAL_DIR.mkdir(exist_ok=True, parents=True)

In [4]:
ismn_data = ISMN_Interface(ISMN_DIR, parallel=True)
crs = "EPSG:4326"
biomass_start = "2025-07-31"

Using the existing ismn metadata in /home/drysat/Downloads/ISMN_archive_20260922/python_metadata/ISMN_archive_20260922.csv to set up ISMN_Interface. 
If there are issues with the data reader, you can remove the metadata csv file to repeat metadata collection.


## Description of the stations

### Split the column based on wether there is an associate depth

In [5]:
meta_full_frame = ismn_data.metadata

is_selected = np.logical_and(
    (meta_full_frame[("network", "val")].isin(list(selection_pairs.keys()))),
    (meta_full_frame[("station", "val")].isin(station_list))
)
within_biomass = (meta_full_frame[("timerange_to", "val")] > biomass_start)
is_moisture = (meta_full_frame[("variable", "val")] == "soil_moisture")

meta_trim_frame = (
    meta_full_frame[is_selected & within_biomass & is_moisture]
    .dropna(how="all", axis=1)
    .drop(columns=["file_path", "file_type"])
    )

# Separate the columns based on wether they have an associate depth
var_col_count = meta_trim_frame.columns.to_series().groupby("variable").size()
is_single_col = (var_col_count == 1)
single_cols = var_col_count[is_single_col].index
multi_cols = var_col_count[np.logical_not(is_single_col)].index

/tmp/ipykernel_23327/532840590.py:13: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  .drop(columns=["file_path", "file_type"])


### Single-depth/spatial data

In [6]:
# Get single station data and 
meta_single_frame = meta_trim_frame[single_cols].droplevel(1, axis=1)

In [7]:
meta_grouped_frame = (
    meta_single_frame.groupby([c for c in meta_single_frame.columns if not c.startswith("timerange")])
    .agg(start=("timerange_from", "min"), end=("timerange_to", "min"))
    .reset_index()
    .set_index(["network", "station"])
    )

In [8]:
points = gpd.points_from_xy(x=meta_grouped_frame["longitude"], y=meta_grouped_frame["latitude"], crs=crs)
spatial_frame = gpd.GeoDataFrame(meta_grouped_frame.drop(columns=["longitude", "latitude"]), geometry=points, crs=crs)

In [9]:
spatial_frame.to_file(WRITE_DIR / "ismn_locations.geojson")

### Multi-depth

In [10]:
# add the network and the station to the multi-frame, clean the columns
meta_multi_frame = meta_trim_frame[multi_cols]
meta_multi_frame[("network", "val")], meta_multi_frame[("station", "val")] = (
    meta_single_frame["network"], meta_single_frame["station"]
)

# Set the index and clean up the name
meta_multi_index_cols = [(c, "val") for c in ("network", "station")]
meta_multi_frame.set_index(meta_multi_index_cols, inplace=True)
meta_multi_frame.index.names = [p[0] for p in meta_multi_frame.index.names]

In [11]:
# Separate instrument columns
instrument_cols = ["instrument", "variable"]
meta_instrument_frame = meta_multi_frame[instrument_cols].dropna(how="all").T.drop_duplicates().T
meta_instrument_frame.columns = ["depth", "sensor", "variable"]

# Separate soil columns 
soil_cols = ["clay_fraction", "sand_fraction", "silt_fraction", "organic_carbon", "saturation"]
meta_soil_frame = meta_multi_frame[soil_cols].dropna(how="all").T.drop_duplicates().T
meta_soil_frame.columns= ["depth_from", "depth_to", "clay_fraction", "sand_fraction", "silt_fraction", "organic_carbon", "saturation"]

In [12]:
meta_instrument_frame.to_csv(WRITE_DIR / "ismn_desc_instruments.csv")
meta_soil_frame.to_csv(WRITE_DIR / "ismn_desc_soil.csv")

## Get the temporal data

In [13]:
temporal_pairs = {}

for network_name, network_stations in selection_pairs.items():

    for station_name in network_stations:

        for sensor in ismn_data[network_name][station_name]:

            sensor_key = (network_name, station_name, sensor.name)
            temporal_pairs[sensor_key] = sensor.data

In [14]:
temporal_frame = (
    pd.concat(temporal_pairs)
    .reset_index(names=["network", "station", "sensor", "date_time"])
    )

In [15]:
is_viable = np.logical_and(
    temporal_frame["date_time"] > biomass_start,
    np.isfinite(temporal_frame["soil_moisture"])
)

tfilt_frame = temporal_frame[is_viable]

In [16]:
is_valid_col = np.logical_not(tfilt_frame.isna().all(axis=0))

In [17]:
tfilt_frame = tfilt_frame.loc[:, is_valid_col[is_valid_col].index]

In [18]:
for station_index, station_frame in tfilt_frame.groupby(["network", "station"]):

    combined_name = "+".join(station_index)
    station_frame.to_csv(TEMPORAL_DIR / f"{combined_name}.csv")